In [19]:
from dotenv import load_dotenv

load_dotenv()
import base64
import json
from flask import Flask, request, jsonify, render_template
import google.generativeai as genai
import os

import PyPDF2 as pdf
from dotenv import load_dotenv
import json

import google.generativeai as genai

In [20]:
load_dotenv() ## load all our environment variables

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

In [26]:
def input_pdf_text(uploaded_file):
    reader=pdf.PdfReader(uploaded_file)
    text=""
    for page in range(len(reader.pages)):
        page=reader.pages[page]
        text+=str(page.extract_text())
    return text

In [27]:

model=genai.GenerativeModel('gemini-2.0-flash')


In [41]:
def get_ats_feedback_GEMINI(resume, job_description):
    text = input_pdf_text(resume)
    
    if not text.strip():
        return "❌ Could not extract text from the PDF. Please check the file."

    # Double curly braces {{}} to escape in .format()
    input_prompt = """
Hey, act like a highly skilled ATS (Applicant Tracking System) specialized in evaluating resumes 
for roles in tech, data science, software engineering, and big data.

### Task:
- Evaluate the candidate **resume** against the given **job description (JD)**
- Score the match as a percentage
- Highlight **strong points** in the resume
- List **missing keywords or skills**
- Provide **specific suggestions for improvement**

Be concise, accurate, and realistic — assume a competitive job market.

Resume:
{text}

Job Description:

{job_description}

### Output format (as a single string, exactly this structure):
{{"ATS Score: <score>%", "Strong points: <...>", "Suggestions: <...>"}} 
Only output the structured string, no extra commentary.
    """.format(resume=text, job_description=job_description)

    model = genai.GenerativeModel('gemini-2.0-flash')
    response = model.generate_content(input_prompt)
    return response.text


In [ ]:
resume="/Users/jasroopsingh/Desktop/resume_ats/jasroop_final_resume (1) (1).pdf"
job_description="""We are looking for a passionate and results-driven Data Scientist to join our AI research and development team. You will work closely with data engineers, ML engineers, and product managers to build predictive models and extract actionable insights from structured and unstructured datasets.
"""
feedback = get_ats_feedback_GEMINI(resume, job_description)
print(feedback)

```json
{"ATS Score: 75%", "Strong points: Strong academic background, relevant AI/ML skills (Python, TensorFlow, PyTorch, scikit-learn, Deep Learning, NLP, Computer Vision), project experience in next word prediction and resume screening, publication in deep learning, Coursera certifications, experience with Git.", "Suggestions: Quantify project results with metrics (e.g., accuracy improvement, efficiency gains). Add specific experience with statistical analysis, data visualization (e.g., matplotlib, seaborn), and cloud computing (AWS, Azure, GCP). Elaborate on experience with feature engineering and model deployment. Mention specific types of data (e.g., time series, image data). Consider adding a brief summary statement at the top highlighting key skills and career interests."}
```


In [ ]:
#USING LLAMA MODEL FROM GROQCLOUD

In [34]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv 
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os

In [35]:
load_dotenv()  # Load environment variables from .env file
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

In [36]:
llm =ChatGroq(
    model_name="llama3-70b-8192",
    groq_api_key=GROQ_API_KEY,
    temperature=0.7
)

In [37]:
response = llm.invoke("What is the capital of india?")
print(response)

content='The capital of India is New Delhi.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 17, 'total_tokens': 26, 'completion_time': 0.025714286, 'prompt_time': 0.00050062, 'queue_time': 0.05389634, 'total_time': 0.026214906}, 'model_name': 'llama3-70b-8192', 'system_fingerprint': 'fp_dd4ae1c591', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None} id='run--485c6766-da9b-489f-974a-48bab200c0b6-0' usage_metadata={'input_tokens': 17, 'output_tokens': 9, 'total_tokens': 26}


In [39]:
def get_ats_feedback_LLAMA(resume, job_description):
    text = input_pdf_text(resume)
    
    if not text.strip():
        return "❌ Could not extract text from the PDF. Please check the file."

    # Double curly braces {{}} to escape in .format()
    prompt = PromptTemplate.from_template(
"""
Hey, act like a highly skilled ATS (Applicant Tracking System) specialized in evaluating resumes 
for roles in tech, data science, software engineering, and big data.

### Task:
- Evaluate the candidate **resume** against the given **job description (JD)**
- Score the match as a percentage
- Highlight **strong points** in the resume
- List **missing keywords or skills**
- Provide **specific suggestions for improvement**

Be concise, accurate, and realistic — assume a competitive job market.

Resume:
{text}

Job Description:
{job_description}

### Output format (as a single string, exactly this structure):
{{"ATS Score: <score>%", "Strong points: <...>", "Suggestions: <...>"}} 
Only output the structured string, no extra commentary.
    """
)
        # Format the prompt
    formatted_prompt = prompt.format(text=text, job_description=job_description)

    # Define the chain
    chain = (
        {"text": lambda _: text, "job_description": lambda _: job_description}
        | prompt
        | llm
        | StrOutputParser()
    )

    # Run the chain
    result = chain.invoke({})

    return result

In [40]:
resume="/Users/jasroopsingh/Desktop/resume_ats/jasroop_final_resume (1) (1).pdf"
job_description="""We are looking for a passionate and results-driven Data Scientist to join our AI research and development team. You will work closely with data engineers, ML engineers, and product managers to build predictive models and extract actionable insights from structured and unstructured datasets.
"""
feedback = get_ats_feedback_LLAMA(resume, job_description)
print(feedback)

{"ATS Score: 72%", "Strong points: Strong educational background, programming skills in Python, C, C++, and SQL, experience with AI/ML libraries and tools, relevant project experience in Next Word Predictor and Resume Screener, leadership and teamwork skills, publication in a conference paper", "Suggestions: Missing keywords: data engineering, product management, cloud platforms, big data tools; add more details on data preprocessing, feature engineering, and model deployment; consider including relevant coursework or academic projects in data science and machine learning; highlight transferable skills from positions of responsibility, such as project management and team leadership"}


In [ ]:



# # 4. Output parser to extract the string
# parser = StrOutputParser()

# # 5. Create the full chain
# chain = prompt | llm | parser

# # 6. Example review
# review_text = "I had a very disappointing experience with Dr. Big. The consultation felt rushed, and I barely had time to explain my symptoms before being cut off. He seemed disinterested and offered little explanation about the diagnosis or treatment. The prescribed medication didn’t improve my condition and led to side effects. I had to follow up multiple times just to get basic clarification. The staff wasn't very helpful either, making the overall visit frustrating. I wouldn’t recommend this clinic based on my experience."



# # 7. Run the chain
# response = chain.invoke({"review": review_text})

# print(response)

In [ ]:
## Flask app setup
app = Flask(__name__)
@app.route('/')
def home():
    return render_template('home.html')




In [ ]:
if __name__ == '__main__':
    app.run(debug=True)  # Run the Flask app in debug mode